In [9]:
# === SESSION BOOTSTRAP ===
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [10]:
!pip install xgboost scikit-learn pandas numpy pyarrow --quiet

In [11]:
# GROUPED-SPLIT robustness: form groups (UAV-NIDD sessions, UAV-CAS configurations) BEFORE dropping them
# from the features, then split at the GROUP level so no session/config spans train / calibration / test.
# Question: do in-distribution coverage and the shift collapse survive grouped splitting?
NIDD={"parquet":"data/uav_nidd/case1.parquet","file":"data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv",
      "label_col":"Label","normal_value":"Normal","subsample_n":200000,
      "families":["DDoS","UDP Flooding","MITM","Jamming","BruteForce","De-authentication"],
      # candidate session-key columns (first available combination is used), tried in order:
      "session_candidates":[["ip.src","ip.dst","srcport","dstport"],["ip.src","ip.dst"],["bssid"],["wlan.sa","wlan.da"]],
      "drops":["unnamed","index","ip.src","ip.dst","ip.proto","wlan.tag","srcport","dstport","udp.srcport","udp.dstport",
           "frame.time","frame.number","time_epoch","time_relative","time_delta","bssid","mactime",
           "vendor_oui","wlan_radio.timestamp","wlan_radio.start_tsf","radiotap.timestamp","wlan.seq","wlan.sa","wlan.da"]}
CAS={"file":"data/uav_cas/UAV-CAS_stat.csv","label_col":"Label","normal_value":"Benign","group_col":"config_idx",
     "single":{"benign","dos","ddos","blackhole","wormhole","replay"},
     "drops":["config_idx","num_drones","num_bs","payload","pathloss","modulation","mission","tx_power","noise","src_ip","dst_ip"]}
CFG={"seeds":[0,1,2,3,4],"alpha":0.10,
     "grp_fracs":{"train":0.60,"cal":0.20,"id_test":0.10,"shift_normal":0.10},
     "xgb":{"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,"colsample_bytree":0.9,"tree_method":"hist"},
     "report_dir":"reports"}
import os; os.makedirs(CFG["report_dir"],exist_ok=True); print("configured")

configured


In [12]:
import numpy as np, pandas as pd, gc, hashlib
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import balanced_accuracy_score
from src.trust import conformal_qhat

def group_split(df, lc, nv, families, held_out, drops, group_ids, seed):
    """Assign each GROUP to a partition; held-out-family records always go to shift.
       train/cal/id_test = seen records (normal+seen attacks) from those groups;
       shift = held-out-family records (all groups) + normal records from shift_normal groups."""
    fam=df[lc].values; y=(fam!=nv).astype(int); g=np.asarray(group_ids)
    seen_fams=[f for f in families if f!=held_out]
    rng=np.random.default_rng(seed)
    uniq=np.unique(g); rng.shuffle(uniq)
    fr=CFG["grp_fracs"]; n=len(uniq); i1=int(fr["train"]*n); i2=i1+int(fr["cal"]*n); i3=i2+int(fr["id_test"]*n)
    gtrain=set(uniq[:i1]); gcal=set(uniq[i1:i2]); gid=set(uniq[i2:i3]); gshift=set(uniq[i3:])
    gof=lambda S: np.array([x in S for x in g])
    seen_mask=np.isin(fam, [nv]+seen_fams)   # records eligible for train/cal/id_test
    tr = seen_mask & gof(gtrain)
    ca = seen_mask & gof(gcal)
    idt= seen_mask & gof(gid)
    sh = (fam==held_out) | ((fam==nv) & gof(gshift))   # shift = held-out attack + normal from shift groups
    # features
    feat=df.drop(columns=[col for col in drops if col in df.columns]+[lc]).apply(pd.to_numeric,errors="coerce")
    feat=feat.drop(columns=[col for col in feat.columns if feat[col].nunique(dropna=False)<=1]).replace([np.inf,-np.inf],np.nan).fillna(0.0)
    X=feat.values.astype(np.float32)
    sc=StandardScaler().fit(X[tr]); tf=lambda m: sc.transform(X[m]).astype(np.float32)
    return {"Xtr":tf(tr),"ytr":y[tr],"Xca":tf(ca),"yca":y[ca],"Xid":tf(idt),"yid":y[idt],
            "Xsh":tf(sh),"ysh":y[sh],"famsh":fam[sh]}, dict(n_groups=n,gtrain=len(gtrain),gcal=len(gcal),gid=len(gid),gshift=len(gshift))
print("grouped splitter ready")

grouped splitter ready


In [13]:
# Build session key for UAV-NIDD from the first available candidate columns
def build_nidd():
    import os
    df=pd.read_parquet(NIDD["parquet"]) if os.path.exists(NIDD["parquet"]) else pd.read_csv(NIDD["file"],low_memory=False,encoding="latin-1")
    if NIDD.get("subsample_n") and len(df)>NIDD["subsample_n"]:
        df=df.groupby(NIDD["label_col"],group_keys=False).sample(frac=NIDD["subsample_n"]/len(df),random_state=42).reset_index(drop=True)
    df=df[df[NIDD["label_col"]].isin([NIDD["normal_value"]]+NIDD["families"])].reset_index(drop=True)
    cols=None
    for cand in NIDD["session_candidates"]:
        if all(col in df.columns for col in cand): cols=cand; break
    if cols is None:
        print("WARNING: no session columns found; falling back to contiguous 200-row bursts as pseudo-sessions")
        gid=(np.arange(len(df))//200)
    else:
        key=df[cols].astype(str).agg("|".join,axis=1)
        gid=key.map({k:i for i,k in enumerate(key.unique())}).values
        print("UAV-NIDD session key from columns:",cols,"->",len(np.unique(gid)),"sessions")
    return df, NIDD["label_col"], NIDD["normal_value"], NIDD["families"], NIDD["drops"], gid

def build_cas():
    raw=pd.read_csv(CAS["file"],low_memory=False); lab=raw[CAS["label_col"]].astype(str)
    keep=lab.str.lower().isin(CAS["single"]) & (~lab.str.contains("+",regex=False)); df=raw[keep].reset_index(drop=True)
    nv=[v for v in df[CAS["label_col"]].unique() if str(v).lower()==CAS["normal_value"].lower()][0]
    fams=[v for v in df[CAS["label_col"]].unique() if v!=nv]
    gid=df[CAS["group_col"]].values
    print("UAV-CAS config groups:",len(np.unique(gid)))
    return df, CAS["label_col"], nv, fams, CAS["drops"], gid
print("dataset builders ready")

dataset builders ready


In [ ]:
def run_grouped(name, df, lc, nv, fams, drops, gid):
    rows=[]
    for F in fams:
        for seed in CFG["seeds"]:
            S,info=group_split(df,lc,nv,fams,F,drops,gid,seed)
            if len(S["yca"])<50 or len(S["ysh"])<50 or S["ytr"].sum()<20: continue
            clf=xgb.XGBClassifier(objective="binary:logistic",eval_metric="logloss",random_state=seed,**CFG["xgb"]).fit(S["Xtr"],S["ytr"])
            qh=conformal_qhat(clf.predict_proba(S["Xca"])[:,1], S["yca"], alpha=CFG["alpha"])
            def covd(X,y):
                p=clf.predict_proba(X)[:,1]; ia=(1-p)<=qh; ino=p<=qh; inset=np.where(y==1,ia,ino)
                return float(inset.mean()), float(np.mean([inset[y==k].mean() for k in np.unique(y)])), balanced_accuracy_score(y,(p>=.5).astype(int))
            id_m,_,_=covd(S["Xid"],S["yid"]); sh_m,sh_b,sh_acc=covd(S["Xsh"],S["ysh"])
            rows.append({"dataset":name,"held_out":str(F),"seed":seed,"id_cov":id_m,"shift_cov_marg":sh_m,"shift_cov_bal":sh_b,"shift_balacc":sh_acc,**info})
            del S,clf; gc.collect()
        print("  ",name,F,"done")
    return pd.DataFrame(rows)

dfc,lcc,nvc,famc,drpc,gidc=build_cas(); Rc=run_grouped("UAV-CAS",dfc,lcc,nvc,famc,drpc,gidc); del dfc; gc.collect()
dfn,lcn,nvn,famn,drpn,gidn=build_nidd(); Rn=run_grouped("UAV-NIDD",dfn,lcn,nvn,famn,drpn,gidn); del dfn; gc.collect()
G=pd.concat([Rc,Rn],ignore_index=True); G.to_csv(os.path.join(CFG["report_dir"],"18_grouped_split_raw.csv"),index=False)
print("rows:",len(G))

UAV-CAS config groups: 1024
   UAV-CAS DoS done
   UAV-CAS DDoS done
   UAV-CAS Blackhole done
   UAV-CAS Wormhole done
   UAV-CAS Replay done
UAV-NIDD session key from columns: ['ip.src', 'ip.dst'] -> 292 sessions


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th

   UAV-NIDD DDoS done


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th

   UAV-NIDD UDP Flooding done


...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th

   UAV-NIDD Jamming done


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use th

   UAV-NIDD BruteForce done


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


In [ ]:
# Grouped-split summary + comparison to the ungrouped panels
summ=G.groupby(["dataset","held_out"]).agg(id_cov=("id_cov","mean"),shift_cov_marg=("shift_cov_marg","mean"),
      shift_cov_bal=("shift_cov_bal","mean"),shift_balacc=("shift_balacc","mean")).round(3)
print("=== GROUPED-SPLIT results (session/config grouping) ===")
print(summ.to_string())
print("\nMean in-distribution coverage under grouped splits: UAV-CAS %.3f, UAV-NIDD %.3f"%(
    G[G.dataset=='UAV-CAS']['id_cov'].mean(), G[G.dataset=='UAV-NIDD']['id_cov'].mean()))
summ.to_csv(os.path.join(CFG["report_dir"],"18_grouped_split_summary.csv"))
# load ungrouped for side-by-side (UAV-NIDD from Table 6 panel raw, UAV-CAS from Table 11 panel raw)
try:
    un=pd.read_csv("reports/14_uavcas_panel_raw.csv").groupby("held_out")["shift_cov_marg"].mean().round(3)
    print("\nUAV-CAS shift coverage, ungrouped vs grouped:")
    cmp=pd.DataFrame({"ungrouped":un,"grouped":Rc.groupby("held_out")["shift_cov_marg"].mean().round(3)})
    print(cmp.to_string())
except Exception as e: print("compare skipped:",e)

In [ ]:
!git add reports/ notebooks/
!git commit -m "18 grouped-split robustness: session-grouped (UAV-NIDD) and configuration-grouped (UAV-CAS) held-out splits; ID coverage and shift collapse under group-level splitting"
!git push origin main